# 🔍 Анализ рынка аналитических вакансий (HH.ru)

**Автор:** Виктор Кобцев  
**Дата создания:** 13.06.2026  
**Версия:** 1.0

---

## 📋 Описание проекта

Автономный ETL-пайплайн для мониторинга вакансий на HH.ru.  
Парсер ежедневно собирает данные, сохраняет исторические "слепки" в локальную базу DuckDB,  
на основе которых строится аналитический дашборд в Apache Superset.

**Цели проекта:**
- Анализ динамики рынка аналитических вакансий
- Мониторинг востребованных навыков и технологий
- Исследование зарплатных диапазонов по регионам и уровням

---

## 🗂️ Структура проекта

```
hh_analitics/
├── data/          # База данных DuckDB
├── notebooks/     # Jupyter ноутбуки
│   └── 01_parser.ipynb
└── logs/          # Логи парсера
```

## 1. Импорт библиотек

In [39]:
import re
import pandas as pd
import duckdb
from datetime import datetime
import time
import json
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from itertools import product

print("Все библиотеки загружены успешно!")
print(f"Pandas версия: {pd.__version__}")
print(f"DuckDB версия: {duckdb.__version__}")
print(f"Текущее время: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Все библиотеки загружены успешно!
Pandas версия: 2.3.3
DuckDB версия: 1.4.4
Текущее время: 2026-06-18 17:50:10


## 2. Настройка параметров

In [40]:
# Базовый URL для поиска
BASE_URL = "https://hh.ru/search/vacancy"

# Пути
DB_PATH = "../data/hh_vacancies.duckdb"
LOG_PATH = "../logs/parser.log"

# Параметры поиска — декартово произведение сегментов
AREAS = ["1", "113&excluded_area=1"]
EXPERIENCE = ["noExperience", "between1And3", "between3And6", "moreThan6"]
WORK_FORMATS = ["ON_SITE", "REMOTE&work_format=HYBRID&work_format=FIELD_WORK"]
HAS_SALARY = ["true", "false"]

SEGMENTS = list(product(AREAS, EXPERIENCE, WORK_FORMATS, HAS_SALARY))
# SEGMENTS = [("1", "between1And3", "ON_SITE", "true")]  #test

print(f"Всего сегментов: {len(SEGMENTS)}")
for i, (area, exp, fmt, has_salary) in enumerate(SEGMENTS):
    print(f"  {i+1}. area={area} | exp={exp} | format={fmt} | has_salary={has_salary}")

Всего сегментов: 32
  1. area=1 | exp=noExperience | format=ON_SITE | has_salary=true
  2. area=1 | exp=noExperience | format=ON_SITE | has_salary=false
  3. area=1 | exp=noExperience | format=REMOTE&work_format=HYBRID&work_format=FIELD_WORK | has_salary=true
  4. area=1 | exp=noExperience | format=REMOTE&work_format=HYBRID&work_format=FIELD_WORK | has_salary=false
  5. area=1 | exp=between1And3 | format=ON_SITE | has_salary=true
  6. area=1 | exp=between1And3 | format=ON_SITE | has_salary=false
  7. area=1 | exp=between1And3 | format=REMOTE&work_format=HYBRID&work_format=FIELD_WORK | has_salary=true
  8. area=1 | exp=between1And3 | format=REMOTE&work_format=HYBRID&work_format=FIELD_WORK | has_salary=false
  9. area=1 | exp=between3And6 | format=ON_SITE | has_salary=true
  10. area=1 | exp=between3And6 | format=ON_SITE | has_salary=false
  11. area=1 | exp=between3And6 | format=REMOTE&work_format=HYBRID&work_format=FIELD_WORK | has_salary=true
  12. area=1 | exp=between3And6 | format=R

## 3. Парсинг вакансий с HH.ru

In [41]:
def create_driver():
    """Создаёт и настраивает браузер Chrome"""
    
    options = Options()
    options.add_argument("--headless")           # без графического интерфейса
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=ru-RU")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
    
    driver = webdriver.Chrome(options=options)
    print("Браузер запущен в фоновом режиме")
    return driver

# Проверяем
driver = create_driver()
print(f"Статус: OK")
driver.quit()
print("Браузер закрыт")

Браузер запущен в фоновом режиме
Статус: OK
Браузер закрыт


In [42]:
def get_salary(card):
    """Извлекает сырой текст зарплаты"""
    spans = card.find_all("span")
    for span in spans:
        text = span.get_text(strip=True)
        if "₽" in text and len(text) < 60 and len(span.find_all("span")) == 0:
            return text
    return None

def parse_salary(salary_raw):
    """Парсит зарплату: извлекает все числа, min -> from, max -> to"""
    if not salary_raw:
        return None, None, None
    
    # Определяем валюту
    currency = "RUB" if "₽" in salary_raw else None
    
    # Извлекаем все числа (убираем пробелы внутри чисел типа "15 000")
    numbers = re.findall(r'\d[\d\s]*\d|\d+', salary_raw)
    numbers = [int(n.replace(' ', '').replace('\u202f', '').replace('\xa0', '')) for n in numbers]
    
    # Фильтруем явно нереалистичные значения (например, год в тексте)
    numbers = [n for n in numbers if 1000 <= n <= 10_000_000]
    
    if not numbers:
        return None, None, currency
    
    salary_from = min(numbers)
    salary_to = max(numbers)
    
    return salary_from, salary_to, currency

def parse_vacancy_card(card):
    """Извлекает данные из одной карточки вакансии"""
    try:
        # Название
        title = card.find("a", {"data-qa": "serp-item__title"})
        title = title.text.strip() if title else None

        # Ссылка
        link = card.find("a", {"data-qa": "serp-item__title"})
        link = link.get("href") if link else None

        # Компания
        company = card.find(attrs={"data-qa": "vacancy-serp__vacancy-employer"})
        company = company.text.strip() if company else None

        # Город
        city = card.find(attrs={"data-qa": "vacancy-serp__vacancy-address"})
        city = city.text.strip() if city else None

        # Опыт
        exp = card.find(attrs={"data-qa": lambda x: x and "work-experience" in x})
        exp = exp.text.strip() if exp else None

        # Удалёнка
        remote = card.find(attrs={"data-qa": lambda x: x and "work-schedule-remote" in x})
        is_remote = True if remote else False

        # Отклик
        responded = card.find(attrs={"data-qa": "vacancy-serp__vacancy_responded"})
        is_applied = True if responded else False

        # Зарплата
        salary_raw = get_salary(card)
        salary_from, salary_to, salary_currency = parse_salary(salary_raw)

        return {
            "title": title,
            "company": company,
            "city": city,
            "experience": exp,
            "is_remote": is_remote,
            "is_applied": is_applied,
            "salary_raw": salary_raw,
            "salary_from": salary_from,
            "salary_to": salary_to,
            "salary_currency": salary_currency,
            "url": link,
            "snapshot_date": datetime.now().strftime("%Y-%m-%d"),
        }

    except Exception as e:
        print(f"Ошибка парсинга карточки: {e}")
        return None

print("Функции парсинга обновлены!")

Функции парсинга обновлены!


In [43]:
def fetch_vacancies(segments):
    """Собирает вакансии по всем сегментам (регион × опыт × формат × зарплата)"""
    
    driver = create_driver()
    all_vacancies = []
    
    try:
        for i, (area, experience, work_format, has_salary) in enumerate(segments):
            print(f"\n🔍 Сегмент {i+1}/{len(segments)}: area={area} | exp={experience} | format={work_format} | has_salary={has_salary}")
            
            for page in range(40):
                url = (
                    f"{BASE_URL}"
                    f"?text=аналитик+OR+analyst+OR+analytics"
                    f"&search_field=name"
                    f"&area={area}"
                    f"&experience={experience}"
                    f"&work_format={work_format}"
                    f"&salary=1"
                    f"&only_with_salary={has_salary}"
                    f"&page={page}"
                )
                
                driver.get(url)
                time.sleep(2)
                
                soup = BeautifulSoup(driver.page_source, "lxml")
                cards = soup.find_all("div", {"data-qa": "vacancy-serp__vacancy"})
                
                if not cards:
                    print(f"  Страница {page + 1}: вакансии не найдены, следующий сегмент")
                    break
                
                for card in cards:
                    vacancy = parse_vacancy_card(card)
                    if vacancy:
                        all_vacancies.append(vacancy)
                
                print(f"  Страница {page + 1}: собрано {len(cards)} вакансий")
                time.sleep(1)
    
    finally:
        driver.quit()
    
    print(f"\nИтого собрано: {len(all_vacancies)} вакансий")
    return all_vacancies

In [44]:
# # Смотрим на данные
# df = pd.DataFrame(raw_vacancies)
# print(f"Размер датафрейма: {df.shape}")
# print(f"\nКолонки: {list(df.columns)}")
# print(f"\nПервые 3 строки:")
# df.head(3)

In [45]:
# df.describe()

## 4. Сохранение в базу данных

In [ ]:
def save_to_db(df, db_path):
    con = duckdb.connect(db_path)
    
    # con.execute("DROP TABLE IF EXISTS vacancies")  # !!!Удаление базы!!!
    
    df = df.drop_duplicates(subset=['url'])
    
    con.execute("""
        CREATE TABLE IF NOT EXISTS vacancies (
            title VARCHAR,
            company VARCHAR,
            city VARCHAR,
            experience VARCHAR,
            is_remote BOOLEAN,
            is_applied BOOLEAN,
            salary_raw VARCHAR,
            salary_from DOUBLE,
            salary_to DOUBLE,
            salary_currency VARCHAR,
            url VARCHAR,
            snapshot_date DATE,
            date_last_seen DATE
        )
    """)
    
    # Новые вакансии — которых ещё нет в базе
    con.execute("""
        INSERT INTO vacancies 
        SELECT *, snapshot_date as date_last_seen FROM df
        WHERE url NOT IN (SELECT url FROM vacancies)
    """)
    
    # Обновляем date_last_seen для уже существующих
    con.execute("""
        UPDATE vacancies
        SET date_last_seen = CURRENT_DATE
        WHERE url IN (SELECT url FROM df)
    """)
    
    count = con.execute("SELECT COUNT(*) FROM vacancies").fetchone()[0]
    today = con.execute("SELECT COUNT(*) FROM vacancies WHERE date_last_seen = CURRENT_DATE").fetchone()[0]
    print(f"Всего записей в базе: {count}")
    print(f"Активных сегодня: {today}")
    
    con.close()
    print("Данные сохранены!")

## 5. Запуск полного пайплайна

In [47]:
import logging

def setup_logger(log_path):
    """Настраивает логгер"""
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[
            logging.FileHandler(log_path, encoding="utf-8"),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

def run_pipeline():
    """Главная функция — запускает весь ETL пайплайн"""
    logger = setup_logger(LOG_PATH)
    logger.info("=== Запуск пайплайна ===")
    
    try:
        # Extract
        logger.info("Шаг 1: Парсинг вакансий...")
        raw_vacancies = fetch_vacancies(SEGMENTS)
        logger.info(f"Собрано вакансий: {len(raw_vacancies)}")
        
        # Transform
        logger.info("Шаг 2: Формирование датафрейма...")
        df = pd.DataFrame(raw_vacancies)
        df = df.dropna(subset=["title"])
        logger.info(f"Датафрейм: {df.shape[0]} строк, {df.shape[1]} колонок")
        
        # Load
        logger.info("Шаг 3: Сохранение в базу данных...")
        save_to_db(df, DB_PATH)
        
        logger.info("=== Пайплайн завершён успешно ===")
        return df
    
    except Exception as e:
        logger.error(f"Ошибка пайплайна: {e}")
        raise


In [48]:
# # Очищаем базу для чистого старта
# con = duckdb.connect(DB_PATH)
# con.execute("DELETE FROM vacancies")
# count = con.execute("SELECT COUNT(*) FROM vacancies").fetchone()[0]
# print(f"Записей в базе после очистки: {count}")
# con.close()

In [49]:
# Ручной запуск: только парсинг (без записи в БД)
logger = setup_logger(LOG_PATH)
raw_vacancies = fetch_vacancies(SEGMENTS)
df = pd.DataFrame(raw_vacancies)
df = df.dropna(subset=["title"])
logger.info(f"Датафрейм готов: {df.shape[0]} строк")

Браузер запущен в фоновом режиме

🔍 Сегмент 1/32: area=1 | exp=noExperience | format=ON_SITE | has_salary=true
  Страница 1: собрано 49 вакансий
  Страница 2: вакансии не найдены, следующий сегмент

🔍 Сегмент 2/32: area=1 | exp=noExperience | format=ON_SITE | has_salary=false
  Страница 1: собрано 50 вакансий
  Страница 2: собрано 50 вакансий
  Страница 3: собрано 11 вакансий
  Страница 4: вакансии не найдены, следующий сегмент

🔍 Сегмент 3/32: area=1 | exp=noExperience | format=REMOTE&work_format=HYBRID&work_format=FIELD_WORK | has_salary=true
  Страница 1: собрано 47 вакансий
  Страница 2: вакансии не найдены, следующий сегмент

🔍 Сегмент 4/32: area=1 | exp=noExperience | format=REMOTE&work_format=HYBRID&work_format=FIELD_WORK | has_salary=false
  Страница 1: собрано 50 вакансий
  Страница 2: собрано 50 вакансий
  Страница 3: собрано 4 вакансий
  Страница 4: вакансии не найдены, следующий сегмент

🔍 Сегмент 5/32: area=1 | exp=between1And3 | format=ON_SITE | has_salary=true
  Страница

2026-06-18 18:33:15,445 | INFO | Датафрейм готов: 16429 строк



Итого собрано: 16429 вакансий


In [50]:
# Ручной запуск: только запись в БД
save_to_db(df, DB_PATH)
logger.info("=== Данные сохранены ===")

2026-06-18 18:33:15,858 | INFO | === Данные сохранены ===


Всего записей в базе: 6869
Активных сегодня: 6869
Данные сохранены!


In [51]:
# print(df)

In [52]:
INCLUDE_WORDS = ['аналитик', 'analyst', 'analytics', 'аналитика']
mask = df['title'].str.lower().str.contains('|'.join(INCLUDE_WORDS))
print(f"Содержит: {mask.sum()} ({mask.sum()/len(df)*100:.1f}%)")
print(f"Не содержит: {(~mask).sum()} ({(~mask).sum()/len(df)*100:.1f}%)")

Содержит: 16429 (100.0%)
Не содержит: 0 (0.0%)


In [53]:
print(f"Всего строк: {df.shape[0]}")
print(f"Уникальных URL: {df['url'].nunique()}")
print(f"Дублей: {df.shape[0] - df['url'].nunique()}")

Всего строк: 16429
Уникальных URL: 6869
Дублей: 9560


In [54]:
# keyword = "аналитик OR analyst OR analytics"
# area = "113"
# page = 0
# url = f"https://hh.ru/search/vacancy?text={keyword}&area={area}&page={page}"
# print(url)

In [55]:
display(df)

,title,company,city,experience,is_remote,is_applied,salary_raw,salary_from,salary_to,salary_currency,url,snapshot_date
0,Бизнес-аналитик,АйСИСОЛЛ,Москва,Без опыта,False,False,"50 000 – 80 000₽за месяц,до вычета налогов",50000.0,80000.0,RUB,https://volgodonsk.hh.ru/vacancy/134163602?que...,2026-06-18
1,Маркетолог (аналитика),Prime Недвижимость,Москва,Без опыта,False,False,"65 000 – 80 000₽за месяц,на руки",65000.0,80000.0,RUB,https://volgodonsk.hh.ru/vacancy/134093419?que...,2026-06-18
2,Стажер группы маркетинговой аналитики,Level Group,Москва,Без опыта,False,False,"от50 000₽за месяц,до вычета налогов",50000.0,50000.0,RUB,https://volgodonsk.hh.ru/vacancy/134320458?que...,2026-06-18
3,Аналитик данных,ООО СиМП Телеком,Москва,Без опыта,False,False,"от90 000₽за месяц,до вычета налогов",90000.0,90000.0,RUB,https://volgodonsk.hh.ru/vacancy/133777412?que...,2026-06-18
4,Аналитик Excel в финансовый департамент,"Перспектива, Кадровый Центр",Москва,Без опыта,False,False,"до120 000₽за месяц,на руки",120000.0,120000.0,RUB,https://volgodonsk.hh.ru/vacancy/134295457?que...,2026-06-18
...,...,...,...,...,...,...,...,...,...,...,...,...
16424,Fraud Analytics [Senior / Lead],ООО Диплей,Омск,Опыт более 6 лет,True,False,None,NaN,NaN,None,https://volgodonsk.hh.ru/vacancy/133878191?que...,2026-06-18
16425,Ведущий BI-аналитик (Senior),Maxim technology,Санкт-Петербург,Опыт более 6 лет,True,False,None,NaN,NaN,None,https://volgodonsk.hh.ru/vacancy/133875915?que...,2026-06-18
16426,Ведущий BI-аналитик (Senior),Maxim technology,"Санкт-Петербург, р-н Центральный",Опыт более 6 лет,True,False,None,NaN,NaN,None,https://volgodonsk.hh.ru/vacancy/133145316?que...,2026-06-18
16427,Head of Analytics (LATAM),ООО Кью Лид,Санкт-Петербург,Опыт более 6 лет,True,False,None,NaN,NaN,None,https://volgodonsk.hh.ru/vacancy/133602430?que...,2026-06-18
